# 03 — KG Embeddings & District Clustering

Train TransE embeddings with PyKEEN, then cluster district vectors into lifestyle zones.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from kg.config import (
    BASE_URI, NUM_DISTRICTS, EMBEDDINGS_OUTPUT, CLUSTERS_OUTPUT,
    CLUSTER_K_RANGE, DISTRICTS_GEOJSON,
)
from kg.embeddings import export_triples, train_embeddings, cluster_districts, export_results

%matplotlib inline

## 1. Train Embeddings

In [ ]:
triples = export_triples()
print(f'{len(triples)} triples extracted for embedding')
print(f'Sample: {triples[0]}')

embeddings, tf = train_embeddings(triples)
print(f'{len(embeddings)} entity embeddings learned')

## 2. t-SNE Visualization

In [ ]:
district_ids = list(range(1, NUM_DISTRICTS + 1))
X = np.array([embeddings[f'{BASE_URI}district/{d}'] for d in district_ids])

tsne = TSNE(n_components=2, random_state=42, perplexity=min(8, len(district_ids) - 1))
X_2d = tsne.fit_transform(X)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(X_2d[:, 0], X_2d[:, 1], s=80, c='steelblue', edgecolors='white', linewidths=0.5)
for i, d in enumerate(district_ids):
    ax.annotate(str(d), (X_2d[i, 0], X_2d[i, 1]), fontsize=10, ha='center', va='bottom', fontweight='bold')
ax.set_title('t-SNE of District Embeddings')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

## 3. Silhouette Analysis

In [ ]:
silhouette_scores = {}
for k in CLUSTER_K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels)
    silhouette_scores[k] = score
    print(f'k={k}: silhouette={score:.4f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(silhouette_scores.keys(), silhouette_scores.values(), color='steelblue')
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score vs. k')
plt.tight_layout()
plt.show()

## 4. Cluster Map

In [ ]:
assignments, best_k = cluster_districts(embeddings)
export_results(embeddings, assignments, best_k)

print(f'Best k: {best_k}')
for d in sorted(assignments):
    print(f'  District {d:2d} -> Cluster {assignments[d]}')

In [ ]:
# t-SNE colored by cluster
COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
cluster_colors = [COLORS[assignments[d]] for d in district_ids]

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(X_2d[:, 0], X_2d[:, 1], s=120, c=cluster_colors, edgecolors='white', linewidths=1)
for i, d in enumerate(district_ids):
    ax.annotate(str(d), (X_2d[i, 0], X_2d[i, 1]), fontsize=10, ha='center', va='bottom', fontweight='bold')
ax.set_title(f'District Clusters (k={best_k})')
plt.tight_layout()
plt.show()

In [ ]:
# Geographic cluster map
districts_gdf = gpd.read_file(str(DISTRICTS_GEOJSON))
for col in ['BEZNR', 'BEZ', 'DISTRICT_CODE', 'district']:
    if col in districts_gdf.columns:
        districts_gdf['d_num'] = districts_gdf[col].astype(int)
        break

districts_gdf['cluster'] = districts_gdf['d_num'].map(assignments)

fig, ax = plt.subplots(figsize=(10, 10))
districts_gdf.plot(column='cluster', ax=ax, categorical=True, cmap='Set2',
                   legend=True, edgecolor='white', linewidth=1.5)
for _, row in districts_gdf.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(str(row.d_num), (centroid.x, centroid.y),
                ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_title(f'Vienna District Clusters (k={best_k})')
ax.axis('off')
plt.tight_layout()
plt.show()